# AML Graph Intelligence Engine — Notebook 3

>We detect everything using *structure only*. The label (`account_suspicious`) and the injected-ring marker are touched **only at the very end** to measure how well our structural hunt worked. We never let them influence detection.

## Step 0 — Load everything built from the previous notebook

We load the two graphs and the feature table we saved, plus re-load the raw transactions.

In [2]:
import os, pickle, time, warnings, itertools
from collections import defaultdict
import pandas as pd, numpy as np, networkx as nx
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120); pd.set_option("display.width", 170)

from google.colab import drive
drive.mount("/content/drive")
DATA_DIR   = "/content/drive/MyDrive/STUDENT_DATASET"
OUTPUT_DIR = os.path.join(DATA_DIR, "outputs")

SENDER_COL, RECEIVER_COL = "Sender_account", "Receiver_account"
AMOUNT_COL, DATE_COL, TIME_COL = "amount_local_npr", "Date", "Time"
ACCOUNT_ID_COL, LABEL_COL = "account_id", "is_suspicious_tx"

# load saved graphs + features
with open(os.path.join(OUTPUT_DIR, "graph_agg.gpickle"),   "rb") as f: G  = pickle.load(f)
with open(os.path.join(OUTPUT_DIR, "graph_multi.gpickle"), "rb") as f: MG = pickle.load(f)
feat = pd.read_csv(os.path.join(OUTPUT_DIR, "node_features.csv"))

# re-load transactions WITH timestamp + label
df_txn      = pd.read_csv(os.path.join(DATA_DIR, "transactions.csv"))
df_features = pd.read_csv(os.path.join(DATA_DIR, "ml_features.csv"))
df_txn["ts"] = pd.to_datetime(df_txn[DATE_COL].astype(str)+" "+df_txn[TIME_COL].astype(str), errors="coerce")
df_txn[LABEL_COL] = df_features[LABEL_COL].values
df_txn = df_txn.sort_values("ts").reset_index(drop=True)

print(f"Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Features: {feat.shape}")
print(f"Transactions: {len(df_txn):,}")
feat = feat.set_index(ACCOUNT_ID_COL)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Graph: 65,339 nodes, 50,586 edges
Features: (65339, 45)
Transactions: 100,222


## Step 1 — Pattern 1: Pass-through / rapid forwarding  


In [3]:
FWD_GAP_HOURS = 48
FWD_GAP = FWD_GAP_HOURS * 3600
TOL_LOW, TOL_HIGH = 0.70, 1.05

# pre-build per-account sorted (time_seconds, amount) arrays
def build_map(group_col):
    # account -> (sorted time_seconds array, matching amount array)
    m = {}
    for k, g in df_txn[[group_col, "ts", AMOUNT_COL]].groupby(group_col):
        t = (g["ts"].astype("int64") // 10**9).values
        a = g[AMOUNT_COL].values
        order = np.argsort(t)
        m[k] = (t[order], a[order])
    return m

in_map  = build_map(RECEIVER_COL)
out_map  = build_map(SENDER_COL)

pass_through_ids = feat.index[feat["is_pass_through"] == 1]

fwd_events = {}     # how many outgoing txns look like forwarded money
fwd_min_gap_h = {}  # fastest forward (hours) - smaller = more mule-like
for acct in pass_through_ids:
    if acct not in in_map or acct not in out_map:
        continue
    t_in, a_in   = in_map[acct]
    t_out, a_out = out_map[acct]
    n_fwd = 0; best_gap = np.inf
    j = 0
    for k in range(len(t_out)):
        to, ao = t_out[k], a_out[k]
        # consider incoming txns that arrived before this outgoing, within the gap window
        lo = np.searchsorted(t_in, to - FWD_GAP, "left")
        hi = np.searchsorted(t_in, to, "right")
        matched = False
        for idx in range(lo, hi):
            ratio = ao / a_in[idx] if a_in[idx] > 0 else 0
            if TOL_LOW <= ratio <= TOL_HIGH:
                matched = True
                best_gap = min(best_gap, (to - t_in[idx]) / 3600.0)
        if matched:
            n_fwd += 1
    fwd_events[acct] = n_fwd
    fwd_min_gap_h[acct] = best_gap if np.isfinite(best_gap) else np.nan

feat["fwd_events"]    = pd.Series(fwd_events).reindex(feat.index).fillna(0)
feat["fwd_min_gap_h"] = pd.Series(fwd_min_gap_h).reindex(feat.index)
feat["is_forwarder"]  = (feat["fwd_events"] >= 2).astype(int)

print(f"Accounts that forwarded received money >=2 times (forwarders): "
      f"{int(feat['is_forwarder'].sum()):,}")
print(f"Of those, how many are actually suspicious (validation only): "
      f"{int(feat.loc[feat['is_forwarder']==1,'account_suspicious'].sum())}")
print("\nFastest-forwarding suspicious-looking accounts:")
print(feat[feat['is_forwarder']==1]
      .sort_values('fwd_min_gap_h')[['fwd_events','fwd_min_gap_h','flow_ratio',
                                     'account_suspicious']].head(8))

Accounts that forwarded received money >=2 times (forwarders): 210
Of those, how many are actually suspicious (validation only): 5

Fastest-forwarding suspicious-looking accounts:
            fwd_events  fwd_min_gap_h  flow_ratio  account_suspicious
account_id                                                           
7228171825         2.0       0.053333    0.406332                   0
7080357663         3.0       0.061944    0.008342                   0
1871309831         2.0       0.062222    0.010530                   0
7344045063         2.0       0.086111    0.998387                   0
8913863501         3.0       0.135833    0.463963                   0
4724445469         2.0       0.166389    0.587748                   0
9093631380         2.0       0.169722    3.065507                   0
8054751831         4.0       0.176944    1.434852                   0


## Step 2 — Patterns 2 & 3: Temporal fan-out and fan-in



In [4]:
WINDOW_HOURS = 24
WINDOW = WINDOW_HOURS * 3600

def max_distinct_in_window(times, parties, window):
    # Max number of DISTINCT parties within any `window`-second span. times sorted asc.
    counts = defaultdict(int); distinct = 0; best = 0; left = 0
    for right in range(len(times)):
        p = parties[right]
        if counts[p] == 0: distinct += 1
        counts[p] += 1
        while times[right] - times[left] > window:
            pl = parties[left]; counts[pl] -= 1
            if counts[pl] == 0: distinct -= 1
            left += 1
        if distinct > best: best = distinct
    return best

# fan-out: per sender, distinct RECEIVERS in window
fanout = {}
for s, g in df_txn[[SENDER_COL, RECEIVER_COL, "ts"]].groupby(SENDER_COL):
    g = g.sort_values("ts")
    t = (g["ts"].astype("int64")//10**9).values
    fanout[s] = max_distinct_in_window(t, g[RECEIVER_COL].values, WINDOW)

# fan-in: per receiver, distinct SENDERS in window
fanin = {}
for r, g in df_txn[[SENDER_COL, RECEIVER_COL, "ts"]].groupby(RECEIVER_COL):
    g = g.sort_values("ts")
    t = (g["ts"].astype("int64")//10**9).values
    fanin[r] = max_distinct_in_window(t, g[SENDER_COL].values, WINDOW)

feat["fanout_24h"] = pd.Series(fanout).reindex(feat.index).fillna(0).astype(int)
feat["fanin_24h"]  = pd.Series(fanin).reindex(feat.index).fillna(0).astype(int)

print(f"Fan-out within {WINDOW_HOURS}h  - max: {feat['fanout_24h'].max()}, "
      f"accounts with >=5: {(feat['fanout_24h']>=5).sum():,}")
print(f"Fan-in  within {WINDOW_HOURS}h  - max: {feat['fanin_24h'].max()}, "
      f"accounts with >=5: {(feat['fanin_24h']>=5).sum():,}")
print("\nTop temporal fan-out accounts:")
print(feat.sort_values('fanout_24h', ascending=False)
      [['fanout_24h','out_degree','account_suspicious']].head(6))
print("\nTop temporal fan-in accounts (collectors):")
print(feat.sort_values('fanin_24h', ascending=False)
      [['fanin_24h','in_degree','account_suspicious']].head(6))

Fan-out within 24h  - max: 23, accounts with >=5: 407
Fan-in  within 24h  - max: 21, accounts with >=5: 215

Top temporal fan-out accounts:
            fanout_24h  out_degree  account_suspicious
account_id                                            
382301928           23        23.0                   0
7758268667          22        22.0                   0
22513864            22        25.0                   0
8976725341          22        25.0                   0
3105440237          21        21.0                   0
7281632640          21        21.0                   0

Top temporal fan-in accounts (collectors):
            fanin_24h  in_degree  account_suspicious
account_id                                          
6086421020         21       21.0                   0
3776724367         20       20.0                   0
3625310304         20       20.0                   0
2670254274         20       21.0                   0
4063749322         20       20.0                   0
14917

## Step 3 — Pattern 5: Circular flows (cycles) with amount conservation



In [5]:
scc_nodes = set(feat.index[feat["scc_size"] >= 2])
Gc = G.subgraph(scc_nodes).copy()
print(f"Searching for cycles inside {Gc.number_of_nodes():,} accounts "
      f"({Gc.number_of_edges():,} edges)...")

t0 = time.time()
try:
    raw_cycles = list(nx.simple_cycles(Gc, length_bound=8))
except TypeError:  # older networkx without length_bound
    raw_cycles = [c for c in itertools.islice(nx.simple_cycles(Gc), 200000) if len(c) <= 8]
print(f"Found {len(raw_cycles):,} simple cycles in {time.time()-t0:.1f}s")

cycle_rows = []
nodes_in_cycle = set()
for cyc in raw_cycles:
    if len(cyc) < 2:
        continue
    amts = []
    ok = True
    for i in range(len(cyc)):
        u, v = cyc[i], cyc[(i+1) % len(cyc)]
        if G.has_edge(u, v):
            amts.append(G[u][v]["weight"])
        else:
            ok = False; break
    if not ok or not amts:
        continue
    conservation = min(amts) / max(amts) if max(amts) > 0 else 0
    cycle_rows.append({"length": len(cyc),
                       "conservation": round(conservation, 3),
                       "min_amt": min(amts), "max_amt": max(amts),
                       "accounts": cyc})
    nodes_in_cycle.update(cyc)

cycles_df = pd.DataFrame(cycle_rows).sort_values("conservation", ascending=False)
# a "strong" cycle = money well-conserved around the loop
CONS_THRESH = 0.80
strong_cycle_nodes = set()
for _, r in cycles_df[cycles_df["conservation"] >= CONS_THRESH].iterrows():
    strong_cycle_nodes.update(r["accounts"])

feat["in_cycle"]        = feat.index.isin(nodes_in_cycle).astype(int)
feat["in_strong_cycle"] = feat.index.isin(strong_cycle_nodes).astype(int)

print(f"\nAccounts in ANY cycle: {len(nodes_in_cycle):,}")
print(f"Accounts in a STRONG (>= {CONS_THRESH} conserved) cycle: {len(strong_cycle_nodes):,}")
print("\nTop conserved cycles:")
print(cycles_df.head(8)[["length","conservation","min_amt","max_amt"]])

Searching for cycles inside 1,525 accounts (2,116 edges)...
Found 1,050 simple cycles in 0.1s

Accounts in ANY cycle: 1,525
Accounts in a STRONG (>= 0.8 conserved) cycle: 284

Top conserved cycles:
      length  conservation     min_amt     max_amt
426        2         0.999  1318701.83  1320425.18
430        2         0.998   341588.76   342281.80
1032       2         0.996   645356.43   648081.30
586        2         0.996   723032.50   725582.56
537        2         0.995   405146.95   407193.17
856        2         0.995  1777044.27  1786687.20
6          2         0.995  8437469.90  8483790.50
250        2         0.994  6274671.47  6310691.07


## Step 4 — Pattern 4: Layering chains (time-respecting, amount-conserving)



In [6]:
MIN_CHAIN_LEN   = 4          # nodes (= 3 hops)
MAX_CHAIN_LEN   = 6          # cutoff for path search
CHAIN_CONS      = 0.50       # weakest edge >= 50% of strongest along the chain
MAX_PATHS_PER_COMPONENT = 4000

UG = G.to_undirected()
components = [c for c in nx.connected_components(UG) if 4 <= len(c) <= 60]
print(f"Examining {len(components):,} islands of size 4-60 for chains...")

def edge_first_ts(u, v):
    return pd.Timestamp(G[u][v]["first_ts"])

chains = []
nodes_in_chain = set()
t0 = time.time()
for comp in components:
    sub = G.subgraph(comp)
    sources = [n for n in comp if sub.in_degree(n) == 0 and sub.out_degree(n) > 0]
    sinks   = [n for n in comp if sub.out_degree(n) == 0 and sub.in_degree(n) > 0]
    if not sources or not sinks:
        continue
    seen = 0
    for s in sources:
        for t in sinks:
            for path in nx.all_simple_paths(sub, s, t, cutoff=MAX_CHAIN_LEN-1):
                seen += 1
                if seen > MAX_PATHS_PER_COMPONENT:
                    break
                if len(path) < MIN_CHAIN_LEN:
                    continue
                # time-respecting?
                ts_seq = [edge_first_ts(path[i], path[i+1]) for i in range(len(path)-1)]
                if any(ts_seq[i] > ts_seq[i+1] for i in range(len(ts_seq)-1)):
                    continue
                # amount-conserving?
                amts = [G[path[i]][path[i+1]]["weight"] for i in range(len(path)-1)]
                cons = min(amts)/max(amts) if max(amts) > 0 else 0
                if cons < CHAIN_CONS:
                    continue
                chains.append({"length": len(path), "conservation": round(cons,3),
                               "accounts": path})
                nodes_in_chain.update(path)
            if seen > MAX_PATHS_PER_COMPONENT: break
        if seen > MAX_PATHS_PER_COMPONENT: break

chains_df = pd.DataFrame(chains).sort_values(["length","conservation"], ascending=False) \
            if chains else pd.DataFrame(columns=["length","conservation","accounts"])
feat["in_chain"] = feat.index.isin(nodes_in_chain).astype(int)

print(f"Found {len(chains_df):,} layering chains in {time.time()-t0:.1f}s")
print(f"Accounts participating in a chain: {len(nodes_in_chain):,}")
if len(chains_df):
    print("\nLongest / best-conserved chains:")
    print(chains_df.head(8)[["length","conservation"]])

Examining 8,042 islands of size 4-60 for chains...
Found 8 layering chains in 3.5s
Accounts participating in a chain: 40

Longest / best-conserved chains:
   length  conservation
1       5         0.982
2       5         0.978
5       5         0.976
6       5         0.975
7       5         0.975
4       5         0.971
3       5         0.967
0       5         0.964


## Step 5 — Combine everything into one **graph risk score** per account



In [7]:
def pct(col):
    return feat[col].rank(pct=True).fillna(0)

# flow-through closeness: only meaningful for pass-through accounts (closer to 1.0 = worse)
flow_close = 1 - (feat["flow_ratio"] - 1).abs().clip(upper=2) / 2
flow_close = flow_close.where(feat["is_pass_through"] == 1, 0)

# weights ~ proportional to measured lift (pass-through dominates)
W = {
    "flow_through": 0.26,
    "fanout":       0.15,
    "fanin":        0.13,
    "out_degree":   0.10,
    "in_degree":    0.08,
    "betweenness":  0.08,
    "forwarding":   0.12,
    "pagerank":     0.03,
}

graph_score = (
    W["flow_through"] * flow_close.rank(pct=True) +
    W["fanout"]       * pct("fanout_24h") +
    W["fanin"]        * pct("fanin_24h") +
    W["out_degree"]   * pct("out_degree") +
    W["in_degree"]    * pct("in_degree") +
    W["betweenness"]  * pct("betweenness_approx") +
    W["forwarding"]   * pct("fwd_events") +
    W["pagerank"]     * pct("pagerank")
)

# rare high-confidence patterns get a multiplicative boost
boost = 1.0 + 0.5*feat["in_strong_cycle"] + 0.4*feat["in_chain"] + 0.3*feat["is_forwarder"]
feat["graph_risk_score"] = (graph_score * boost)
# rescale to 0-100 for readability
feat["graph_risk_score"] = (100 * (feat["graph_risk_score"] - feat["graph_risk_score"].min())
                            / (feat["graph_risk_score"].max() - feat["graph_risk_score"].min()))

print("graph_risk_score summary:")
print(feat["graph_risk_score"].describe().round(2))
print("\nTop 10 accounts by graph risk score:")
print(feat.sort_values("graph_risk_score", ascending=False)
      [["graph_risk_score","fanout_24h","fanin_24h","fwd_events","in_chain",
        "in_strong_cycle","account_suspicious","injected_ring"]].head(10))

graph_risk_score summary:
count    65339.00
mean         2.47
std          6.54
min          0.00
25%          0.46
50%          1.01
75%          1.68
max        100.00
Name: graph_risk_score, dtype: float64

Top 10 accounts by graph risk score:
            graph_risk_score  fanout_24h  fanin_24h  fwd_events  in_chain  in_strong_cycle  account_suspicious  injected_ring
account_id                                                                                                                   
6744599788        100.000000           5          5         3.0         0                1                   0              0
6710608627         99.767578           4          3         3.0         0                1                   0              0
3995490280         99.356474           4          3         2.0         0                1                   0              0
8338562032         98.997738           3          4         5.0         0                1                   0             

## Step 6 - Validate against the hidden label



In [8]:
base = feat["account_suspicious"].mean()
print(f"Base rate of suspicious accounts: {base:.3%}\n")
print(f"{'K':>6} | {'precision@K':>11} | {'lift':>6} | {'ring caught':>11}")
print("-"*45)
ranked = feat.sort_values("graph_risk_score", ascending=False)
for K in [50, 100, 200, 500]:
    top = ranked.head(K)
    prec = top["account_suspicious"].mean()
    ring = int(top["injected_ring"].sum())
    print(f"{K:>6} | {prec:>10.2%} | {prec/base:>5.1f}x | {ring:>11}")

total_ring = int(feat["injected_ring"].sum())
total_susp = int(feat["account_suspicious"].sum())
caught_500 = int(ranked.head(500)["injected_ring"].sum())
susp_500   = int(ranked.head(500)["account_suspicious"].sum())
print(f"\nInjected ring: rediscovered {caught_500}/{total_ring} in our top 500 (structure only).")
print(f"Suspicious accounts: {susp_500}/{total_susp} of ALL suspicious accounts are in our top 500.")

Base rate of suspicious accounts: 0.488%

     K | precision@K |   lift | ring caught
---------------------------------------------
    50 |      2.00% |   4.1x |           0
   100 |      1.00% |   2.0x |           0
   200 |      3.50% |   7.2x |           3
   500 |     12.40% |  25.4x |          56

Injected ring: rediscovered 56/171 in our top 500 (structure only).
Suspicious accounts: 62/319 of ALL suspicious accounts are in our top 500.


## Step 7 — Rank the **islands**

In [9]:
comp_rows = []
for comp in nx.connected_components(UG):
    comp = list(comp)
    sub_feat = feat.loc[feat.index.isin(comp)]
    comp_rows.append({
        "n_accounts":     len(comp),
        "max_fanout":     int(sub_feat["fanout_24h"].max()),
        "max_fanin":      int(sub_feat["fanin_24h"].max()),
        "n_forwarders":   int(sub_feat["is_forwarder"].sum()),
        "has_chain":      int(sub_feat["in_chain"].max()),
        "has_strong_cycle": int(sub_feat["in_strong_cycle"].max()),
        "total_flow_npr": float(sub_feat["weighted_out"].sum()),
        "mean_acct_risk": float(sub_feat["graph_risk_score"].mean()),
        "max_acct_risk":  float(sub_feat["graph_risk_score"].max()),
        "n_suspicious":   int(sub_feat["account_suspicious"].sum()),   # validation only
        "accounts":       comp,
    })

subg = pd.DataFrame(comp_rows)
# island risk = blend of its strongest signals (structure only; n_suspicious NOT used)
subg["subgraph_risk"] = (
    0.35*subg["mean_acct_risk"]/100 +
    0.20*(subg["max_fanout"].clip(upper=20)/20) +
    0.15*(subg["max_fanin"].clip(upper=20)/20) +
    0.15*subg["has_chain"] +
    0.15*subg["has_strong_cycle"]
) * 100
subg = subg.sort_values("subgraph_risk", ascending=False).reset_index(drop=True)

print("Top 12 suspicious islands:")
print(subg.head(12)[["n_accounts","max_fanout","max_fanin","n_forwarders",
                     "has_chain","has_strong_cycle","subgraph_risk","n_suspicious"]])

Top 12 suspicious islands:
    n_accounts  max_fanout  max_fanin  n_forwarders  has_chain  has_strong_cycle  subgraph_risk  n_suspicious
0            4           1          1             0          0                 1      36.706970             4
1            4           1          1             0          0                 1      36.695113             4
2            4           1          1             0          0                 1      35.877363             4
3            4           1          1             0          0                 1      35.869068             4
4            4           1          1             0          0                 1      35.867121             4
5            4           1          1             0          0                 1      35.861458             4
6            4           1          1             0          0                 1      35.242458             4
7            2           1          1             0          0                 1      35.2417

## Step 8 — Save the deliverables



In [10]:
# updated features (with all pattern columns)
feat_out = feat.reset_index()
feat_out.to_csv(os.path.join(OUTPUT_DIR, "node_features.csv"), index=False)

# ranked accounts
acct_cols = ["graph_risk_score","fanout_24h","fanin_24h","out_degree","in_degree",
             "fwd_events","fwd_min_gap_h","flow_ratio","in_chain","in_cycle",
             "in_strong_cycle","betweenness_approx","pagerank",
             "pep_flag","sanctions_hit","risk_grade",
             "account_suspicious","injected_ring"]
ranked_accounts = feat.sort_values("graph_risk_score", ascending=False)[acct_cols].reset_index()
ranked_accounts.to_csv(os.path.join(OUTPUT_DIR, "ranked_accounts.csv"), index=False)

# ranked subgraphs (store accounts as a string so CSV is clean)
subg_save = subg.copy()
subg_save["accounts"] = subg_save["accounts"].apply(lambda xs: ";".join(map(str, xs)))
subg_save.to_csv(os.path.join(OUTPUT_DIR, "ranked_subgraphs.csv"), index=False)

# also save the chain & cycle catalogues for the explainability notebook
chains_df.to_pickle(os.path.join(OUTPUT_DIR, "chains.pkl"))
cycles_df.to_pickle(os.path.join(OUTPUT_DIR, "cycles.pkl"))

print("Saved:")
for fn in ["node_features.csv","ranked_accounts.csv","ranked_subgraphs.csv","chains.pkl","cycles.pkl"]:
    print("  ", os.path.join(OUTPUT_DIR, fn))

Saved:
   /content/drive/MyDrive/STUDENT_DATASET/outputs/node_features.csv
   /content/drive/MyDrive/STUDENT_DATASET/outputs/ranked_accounts.csv
   /content/drive/MyDrive/STUDENT_DATASET/outputs/ranked_subgraphs.csv
   /content/drive/MyDrive/STUDENT_DATASET/outputs/chains.pkl
   /content/drive/MyDrive/STUDENT_DATASET/outputs/cycles.pkl
